<a href="https://colab.research.google.com/github/Dwayne-tech/DML/blob/main/Chatbot%20official.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Installing the required packages
!pip install streamlit
!pip install pyngrok
!pip install streamlit-option-menu

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.3/44.3 kB 1.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 34.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 58.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 79.1/79.1 kB 5.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 829.3/829.3 kB 31.8 MB/s eta 0:00:00


In [25]:
%%writefile app.py
# Import necessary libraries
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.ensemble import RandomForestClassifier

# Function to load Netflix data
def load_data():
    # Load Netflix dataset (make sure to upload the CSV file first)
    data = pd.read_csv("netflix_titles.csv")

    # Data preprocessing
    data = data.dropna(subset=['title', 'listed_in', 'description'])
    data['combined_features'] = data['listed_in'] + ' ' + data['description']
    return data

# Function to train recommendation model
def train_model(data):
    # Create TF-IDF Vectorizer
    tfidf = TfidfVectorizer(stop_words='english')
    tfidf_matrix = tfidf.fit_transform(data['combined_features'])

    # Compute cosine similarity matrix
    cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
    return cosine_sim, data

# Function to get recommendations
def get_recommendations(title, cosine_sim, df):
    try:
        # Get index of the movie
        idx = df[df['title'] == title].index[0]

        # Get pairwise similarity scores
        sim_scores = list(enumerate(cosine_sim[idx]))

        # Sort by similarity score
        sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)

        # Get top 10 recommendations
        sim_scores = sim_scores[1:11]
        movie_indices = [i[0] for i in sim_scores]

        return df['title'].iloc[movie_indices]
    except IndexError:
        return None

# Main app function
def main():
    st.title("Netflix Content Explorer")
    st.markdown("Discover shows and get personalized recommendations")

    # Load data and model
    @st.cache_resource
    def load_resources():
        data = load_data()
        cosine_sim, cleaned_data = train_model(data)
        return cosine_sim, cleaned_data

    cosine_sim, netflix_data = load_resources()

    # Create tabs
    tab1, tab2 = st.tabs(["Content Assistant", "Recommendation Engine"])

    with tab1:
        st.header("Netflix Content Q&A")
        question = st.selectbox("Choose a question:", [
            "What's the most common genre?",
            "Which country produces the most content?",
            "What's the average movie duration?"
        ])

        if st.button("Get Answer"):
            if question == "What's the most common genre?":
                common_genre = netflix_data['listed_in'].value_counts().index[0]
                st.write(f"Most common genre: {common_genre}")
            elif question == "Which country produces the most content?":
                top_country = netflix_data['country'].value_counts().index[0]
                st.write(f"Top content producer: {top_country}")
            else:
                movies = netflix_data[netflix_data['type'] == 'Movie']
                avg_duration = movies['duration'].str.extract('(\d+)')[0].astype(float).mean()
                st.write(f"Average movie duration: {avg_duration:.1f} minutes")

    with tab2:
        st.header("Personalized Recommendations")
        title_input = st.selectbox("Select a title:", netflix_data['title'].unique())

        if st.button("Show Recommendations"):
            recommendations = get_recommendations(title_input, cosine_sim, netflix_data)

            if recommendations is not None:
                st.subheader(f"Recommended shows similar to {title_input}:")
                for i, title in enumerate(recommendations, 1):
                    st.write(f"{i}. {title}")
            else:
                st.error("Title not found. Please select a valid title from the dropdown.")

if __name__ == "__main__":
    main()

Overwriting app.py


In [27]:
import time
import subprocess
from pyngrok import ngrok

# Set your Ngrok authtoken
ngrok.set_auth_token("2rtFOBrMcUwbOtZblyreHGz8Ivm_Z3itCyvrSmVaYJPqif51")

# Install required dependencies
subprocess.run(["pip", "install", "-q", "streamlit", "pyngrok", "scikit-learn", "pandas"], check=True)

try:
    # Start Streamlit app in background
    process = subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", "8501", "--server.headless", "true"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
        shell=True
    )

    # Wait for app initialization
    time.sleep(8)

    # Create secure Ngrok tunnel
    public_url = ngrok.connect(8501, "http", bind_tls=True)

    # Display connection information
    print("\n" + "="*50)
    print(f"⚡ Public Access URL: {public_url}")
    print("="*50 + "\n")
    print("Keep this terminal running to maintain access!")
    print("Press Ctrl+C to terminate both server and tunnel\n")

    # Maintain persistent connection
    while True:
        time.sleep(300)  # Refresh connection every 5 minutes
        print("Connection refresh...")  # Heartbeat indicator

except KeyboardInterrupt:
    print("\nTerminating processes...")
    process.kill()
    ngrok.kill()
    print("Cleanup complete!")

except Exception as e:
    print(f"Error occurred: {str(e)}")
    process.kill()
    ngrok.kill()


⚡ Public Access URL: NgrokTunnel: "https://a6b1-34-85-180-183.ngrok-free.app" -> "http://localhost:8501"

Keep this terminal running to maintain access!
Press Ctrl+C to terminate both server and tunnel



Connection refresh...

Terminating processes...
Cleanup complete!


In [32]:
# app.py
import streamlit as st
import pandas as pd
import numpy as np
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import csv

# Load data with enhanced error handling
def load_data():
    try:
        df = pd.read(
            "/content/netflix titles.zip",
            delimiter=',',
            quotechar='"',
            encoding='utf-8',
            on_bad_lines='skip',
            engine='python',
            quoting=csv.QUOTE_MINIMAL
        )
        df = df.dropna(subset=['title', 'listed_in', 'description'])
        df['combined_features'] = df['listed_in'].str.lower() + ' ' + df['description'].str.lower()
        return df
    except Exception as e:
        st.error(f"Data loading error: {str(e)}")
        return pd.DataFrame()

# Recommendation system with validation
def train_model(data):
    try:
        tfidf = TfidfVectorizer(stop_words='english', max_features=5000)
        tfidf_matrix = tfidf.fit_transform(data['combined_features'])
        cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)
        return cosine_sim
    except Exception as e:
        st.error(f"Model training error: {str(e)}")
        return None

def main():
    st.title("Netflix Content Advisor")

    # Load data with caching
    @st.cache_resource
    def load_resources():
        data = load_data()
        model = train_model(data) if not data.empty else None
        return model, data

    model, df = load_resources()

    if df.empty:
        st.error("Failed to load data. Ensure 'netflix_titles.csv' is in the directory.")
        return

    tab1, tab2 = st.tabs(["Analytics", "Recommendations"])

    with tab1:
        st.header("Content Insights")
        if st.button("Show Basic Stats"):
            st.write(f"Total Titles: {len(df)}")
            st.write(f"Movies: {len(df[df['type'] == 'Movie'])}")
            st.write(f"TV Shows: {len(df[df['type'] == 'TV Show'])}")

    with tab2:
        st.header("Personalized Suggestions")
        title = st.selectbox("Choose a title:", df['title'].unique())

        if st.button("Get Recommendations") and model is not None:
            try:
                idx = df[df['title'] == title].index[0]
                sim_scores = list(enumerate(model[idx]))
                sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:11]
                results = df.iloc[[i[0] for i in sim_scores]]['title']
                st.write("Similar titles:")
                st.dataframe(results)
            except Exception as e:
                st.error(f"Error generating recommendations: {str(e)}")

# deploy.py
import time
import subprocess
from pyngrok import ngrok

# Configuration
NGROK_TOKEN = "2rtFOBrMcUwbOtZblyreHGz8Ivm_Z3itCyvrSmVaYJPqif51"
PORT = 8501

def setup():
    # Install dependencies
    subprocess.run(["pip", "install", "-q", "streamlit", "pandas", "scikit-learn", "pyngrok"])

    # Set up Ngrok
    ngrok.set_auth_token(NGROK_TOKEN)

    # Start Streamlit
    process = subprocess.Popen(
        ["streamlit", "run", "app.py", "--server.port", str(PORT), "--server.headless", "true"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL
    )

    time.sleep(8)  # Wait for server initialization

    # Create tunnel
    tunnel = ngrok.connect(PORT, "http", bind_tls=True)
    print(f"\n{'='*50}\nPublic URL: {tunnel.public_url}\n{'='*50}\n")

    try:
        while True:
            time.sleep(10)
    except KeyboardInterrupt:
        print("\nShutting down...")
        process.kill()
        ngrok.kill()

if __name__ == "__main__":
    setup()


Public URL: https://e135-34-85-180-183.ngrok-free.app


Shutting down...
